# 02 · Vocab + emphasis analyzers (modality, vocab, all_caps, caps_imperative, justification)

This is **stage 2 of the 6-stage producer chain**. It reloads the cache from stage 00 and runs five analyzers focused on vocabulary, emphasis, and explanation.

| Analyzer | What it produces |
|---|---|
| `modality_for_doc` | Three-class modality (`deontic` / `epistemic` / `dynamic`) via spaCy parse-tree patterns. |
| `vocab_for_doc` | Counts per `VOCAB` lexicon class (prohibitions, prescriptions, politeness, warmth, hedging, structural, profanity, pronouns). |
| `all_caps_for_text` | ALL CAPS tokens with `TECH_ACRONYMS` excluded. |
| `caps_imperative_for_text` | Direct match against `CAPS_IMPERATIVE_TOKENS` (e.g., `MUST`, `NEVER`, `IMPORTANT`). |
| `justification_for_text` | `JUSTIFICATION_PATTERNS` matches + the `ratio = count / (marker_count + 1)`. |

The justification ratio depends on the per-file imperative `marker_count`. We recompute it locally with the same matcher used in stage 01 (`prompt_pipeline.M_IMPERATIVE`) — keeps stage 02 self-contained without reading stage 01's partial.

Output: `_pipeline_cache/partial_vocab_emphasis.json`.

In [1]:
"""Reload corpus + DocBin from stage 00."""
import os, pathlib, json, importlib
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm
from spacy.tokens import DocBin

PROJECT_ROOT = pathlib.Path("/home/user/workspace/claude-prompts-analysis").resolve()
if pathlib.Path.cwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

CACHE_DIR  = PROJECT_ROOT / "_pipeline_cache"
DOCBIN_IN  = CACHE_DIR / "corpus_docs.spacy"
META_IN    = CACHE_DIR / "corpus_meta.parquet"
PARTIAL_OUT = CACHE_DIR / "partial_vocab_emphasis.json"

assert DOCBIN_IN.exists(),  f"missing {DOCBIN_IN} — run 00_setup_and_corpus first"
assert META_IN.exists(),    f"missing {META_IN} — run 00_setup_and_corpus first"

import prompt_pipeline
importlib.reload(prompt_pipeline)
from prompt_pipeline import NLP

df = pd.read_parquet(META_IN)
docs = list(DocBin().from_disk(DOCBIN_IN).get_docs(NLP.vocab))
assert len(docs) == len(df), f"DocBin/df length mismatch: {len(docs)} vs {len(df)}"
print(f"reloaded {len(df)} files")

reloaded 286 files


## 4. Modality

A single spaCy parse-tree detector classifies every modal expression as **deontic** (`must`, `should`, `have to`, `need to` + verb), **epistemic** (`may`, `might`, modals followed by `be`/`have`, epistemic adverbs like `likely`/`probably`), or **dynamic** (`can`, `could`, `able to`, `will`/`would`).

In [2]:
from prompt_pipeline import modality_for_doc

modality_per_file = [modality_for_doc(d, n, s)
                     for d, n, s in zip(docs, df["n_tokens"], df["n_sents"])]
df_modality = pd.DataFrame(modality_per_file)

print("per-file modality (head):")
print(df_modality.head().to_string())
print()
print("category means (% of words and per-sentence rate):")
key_cols = ["deontic_pct", "deontic_per_sent",
            "epistemic_pct", "epistemic_per_sent",
            "dynamic_pct", "dynamic_per_sent"]
print(pd.concat([df[["category"]], df_modality[key_cols]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())
print()
print("corpus-wide modality counts:")
totals = df_modality[["deontic_count", "epistemic_count", "dynamic_count"]].sum()
print(totals.to_string())

per-file modality (head):
   deontic_count  deontic_pct  deontic_per_sent  epistemic_count  epistemic_pct  epistemic_per_sent  dynamic_count  dynamic_pct  dynamic_per_sent top_construction
0              5       0.5342            0.2273                8         0.8547              0.3636              4       0.4274            0.1818           should
1              2       0.8264            0.1538                2         0.8264              0.1538              2       0.8264            0.1538           should
2              7       0.1981            0.0609                0         0.0000              0.0000             48       1.3586            0.4174              'll
3              1       0.2667            0.0385                1         0.2667              0.0385              2       0.5333            0.0769               ca
4              0       0.0000            0.0000                0         0.0000              0.0000              0       0.0000            0.0000              

## 5. Vocabulary profile

`VOCAB` is an 11-class lexicon: hard / soft prohibitions, hard / soft prescriptions, politeness (direct + softening), warmth, hedging, structural markers, profanity, and 1st/2nd-person pronouns.

In [3]:
from prompt_pipeline import vocab_for_doc, VOCAB_KEYS

vocab_per_file = [vocab_for_doc(d, n, s)
                  for d, n, s in zip(docs, df["n_tokens"], df["n_sents"])]
df_vocab = pd.DataFrame(vocab_per_file)

pct_cols      = [f"{k}_pct" for k in VOCAB_KEYS]
per_sent_cols = [f"{k}_per_sent" for k in VOCAB_KEYS]

print("category mean (% of words):")
print(pd.concat([df[["category"]], df_vocab[pct_cols]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())
print()
print("corpus-wide raw counts:")
total = df_vocab[[f"{k}_count" for k in VOCAB_KEYS]].sum()
for k in VOCAB_KEYS:
    print(f"  {k:24s} {int(total[f'{k}_count']):6d}")

category mean (% of words):
                  hard_prohibitions_pct  hard_prescriptions_pct  soft_prescriptions_pct  politeness_direct_pct  politeness_softening_pct  warmth_encouragement_pct  hedging_pct  structural_markers_pct  profanity_pct  pronouns_2p_pct  pronouns_1p_pct
category                                                                                                                                                                                                                                                
Agent prompt                      0.470                   0.233                   0.381                  0.045                     0.032                     0.004        0.164                   0.230            0.0            1.351            0.093
Data / template                   0.275                   0.284                   0.023                  0.000                     0.039                     0.000        0.078                   0.128            0.0           

## 6. ALL CAPS emphasis

Every uppercase token (≥2 characters) excluding the curated `TECH_ACRONYMS` allowlist (e.g. `API`, `URL`, `JSON` — not emphasis-flagged).

In [4]:
from prompt_pipeline import all_caps_for_text, ALLCAPS_RE, TECH_ACRONYMS

all_caps_per_file = [all_caps_for_text(t, n, s)
                     for t, n, s in zip(df["raw_text"], df["n_tokens"], df["n_sents"])]
df_all_caps = pd.DataFrame(all_caps_per_file)

print("per-file ALL CAPS (head):")
print(df_all_caps.head().to_string())
print()
print("category mean ALL CAPS (% of words / per sentence):")
print(pd.concat([df[["category"]], df_all_caps[["count", "pct", "per_sent"]]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())

corpus_all_caps = Counter()
for txt in df["raw_text"]:
    for tok in ALLCAPS_RE.findall(txt):
        if tok not in TECH_ACRONYMS:
            corpus_all_caps[tok] += 1
print()
print("corpus-wide top-25 ALL CAPS tokens:")
for tok, c in corpus_all_caps.most_common(25):
    print(f"  {tok:20s} {c}")

per-file ALL CAPS (head):
   count  distinct     pct  per_sent                                                                                                        top
0      6         3  0.6410    0.2727  [{'token': 'CLAUDE', 'count': 3}, {'token': 'TASK_TOOL_NAME', 'count': 2}, {'token': 'NOTE', 'count': 1}]
1      0         0  0.0000    0.0000                                                                                                         []
2     40        33  1.1322    0.3478        [{'token': 'GITHUB_TOKEN', 'count': 3}, {'token': 'THE', 'count': 2}, {'token': 'OTP', 'count': 2}]
3      0         0  0.0000    0.0000                                                                                                         []
4      0         0  0.0000    0.0000                                                                                                         []

category mean ALL CAPS (% of words / per sentence):
                   count    pct  per_sent
category       

## 7. CAPS imperative tokens

Direct match against `CAPS_IMPERATIVE_TOKENS` (`IMPORTANT`, `MUST`, `NEVER`, `DO NOT`, `WARNING` …). Word-boundary regex handles multi-word phrases (`MUST NOT`, `VERY IMPORTANT`).

In [5]:
from prompt_pipeline import caps_imperative_for_text, CAPS_IMP_RE, CAPS_IMPERATIVE_TOKENS

caps_imperative_per_file = [
    caps_imperative_for_text(t, n, s)
    for t, n, s in zip(df["raw_text"], df["n_tokens"], df["n_sents"])
]
df_caps_imperative = pd.DataFrame(caps_imperative_per_file)

print("per-file CAPS imperative (head):")
print(df_caps_imperative.head().to_string())
print()
print("category mean CAPS imperative (% of words / per sentence):")
print(pd.concat([df[["category"]], df_caps_imperative[["count", "pct", "per_sent"]]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())

corpus_caps_imperative = Counter()
for txt in df["raw_text"]:
    corpus_caps_imperative.update(CAPS_IMP_RE.findall(txt))
print()
print("corpus-wide CAPS imperative frequency:")
for tok in CAPS_IMPERATIVE_TOKENS:
    if corpus_caps_imperative[tok]:
        print(f"  {tok:18s} {corpus_caps_imperative[tok]}")

per-file CAPS imperative (head):
   count     pct  per_sent         hits
0      1  0.1068    0.0455  {'NOTE': 1}
1      0  0.0000    0.0000           {}
2      0  0.0000    0.0000           {}
3      0  0.0000    0.0000           {}
4      0  0.0000    0.0000           {}

category mean CAPS imperative (% of words / per sentence):
                  count    pct  per_sent
category                                
Agent prompt      0.919  0.168     0.062
Data / template   0.028  0.002     0.000
Skill             0.367  0.013     0.004
System prompt     0.381  0.161     0.048
System reminder   0.425  0.224     0.054
Tool description  0.608  0.348     0.079
Tool parameter    0.000  0.000     0.000

corpus-wide CAPS imperative frequency:
  IMPORTANT          35
  VERY IMPORTANT     1
  CRITICAL           15
  MANDATORY          2
  REQUIRED           2
  MUST               18
  MUST NOT           3
  NEVER              25
  ALWAYS             7
  DO NOT             14
  NOTE               7


## 8. Justification patterns

Counts of `JUSTIFICATION_PATTERNS` regex matches plus the **justification ratio**: `count / (marker_count + 1)` per file. The `+1` prevents division-by-zero when a file contains zero imperative markers (and softens the ratio for very-low-marker files). We recompute `marker_count` locally so this notebook doesn't depend on stage 01's partial.

In [6]:
from prompt_pipeline import justification_for_text, M_IMPERATIVE, count_matcher

# Recompute marker_count per file (deterministic; same matcher as stage 01).
marker_counts = [count_matcher(d, M_IMPERATIVE["imperative_markers"]) for d in docs]

justification_per_file = [
    justification_for_text(t, n, s, m)
    for t, n, s, m in zip(df["clean_text"], df["n_tokens"], df["n_sents"], marker_counts)
]
df_justification = pd.DataFrame(justification_per_file)

print("per-file justification (head):")
print(df_justification.head().to_string())
print()
print("category mean (justification, % of words / per sentence):")
print(pd.concat([df[["category"]], df_justification], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())

per-file justification (head):
   count     pct  per_sent  ratio
0      6  0.6410    0.2727  0.857
1      1  0.4132    0.0769  1.000
2      7  0.1981    0.0609  0.412
3      2  0.5333    0.0769  0.400
4      1  0.5556    0.2000  0.500

category mean (justification, % of words / per sentence):
                  count    pct  per_sent  ratio
category                                       
Agent prompt      1.892  0.324     0.085  0.337
Data / template   1.000  0.101     0.020  0.198
Skill             2.633  0.273     0.083  0.377
System prompt     0.825  0.292     0.073  0.242
System reminder   0.425  0.255     0.054  0.101
Tool description  0.734  0.347     0.078  0.150
Tool parameter    0.000  0.000     0.000  0.000


## Write `partial_vocab_emphasis.json`

In [7]:
partial = {
    str(i): {
        "modality":         modality_per_file[i],
        "vocab":            vocab_per_file[i],
        "all_caps":         all_caps_per_file[i],
        "caps_imperative":  caps_imperative_per_file[i],
        "justification":    justification_per_file[i],
    }
    for i in range(len(df))
}
with open(PARTIAL_OUT, "w") as f:
    json.dump(partial, f)
size = PARTIAL_OUT.stat().st_size
print(f"wrote {PARTIAL_OUT.relative_to(PROJECT_ROOT)}  ({size:,} bytes, {size/1024:.1f} KiB)")
print(f"      {len(partial)} per-file records, 5 blocks each")

wrote _pipeline_cache/partial_vocab_emphasis.json  (446,584 bytes, 436.1 KiB)
      286 per-file records, 5 blocks each
